# 03. Positional Encoding

## 학습 목표
- 왜 Transformer에 위치 정보가 필요한지 이해
- Sinusoidal PE를 수식부터 구현, 시각화까지
- Learned PE, RoPE, ALiBi 각 방식 이해 및 비교
- 현대 LLM에서 어떤 방식을 쓰는지 파악

## 참고 자료
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) - Sinusoidal PE
- [RoFormer](https://arxiv.org/abs/2104.09864) - Rotary Position Embedding
- [ALiBi](https://arxiv.org/abs/2108.12409) - Attention with Linear Biases

---

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)

## 1. 왜 위치 정보가 필요한가?

RNN은 순서대로 처리하므로 자연스럽게 위치 정보를 가진다.
하지만 **Transformer는 모든 토큰을 동시에 처리** → 순서 정보가 없다.

위치 정보가 없으면 아래 두 문장을 구분할 수 없다:
- "개가 사람을 물었다" (개 → 물다의 주어)
- "사람이 개를 물었다" (사람 → 물다의 주어)

Attention 연산 자체는 **집합(Set)에 대한 연산**이므로 순서를 바꿔도 결과가 같다.

In [ ]:
# 순서를 바꿔도 Self-Attention 결과가 같다는 것을 확인

import torch.nn.functional as F

def simple_self_attention(x):
    """위치 정보 없는 단순 self-attention"""
    d_k = x.size(-1)
    scores = torch.matmul(x, x.transpose(-2, -1)) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, x)

# 3개 토큰: A, B, C
torch.manual_seed(0)
A = torch.randn(1, 4)
B = torch.randn(1, 4)
C = torch.randn(1, 4)

# 순서 1: [A, B, C]
x1 = torch.stack([A, B, C], dim=1)  # (1, 3, 4)
out1 = simple_self_attention(x1)

# 순서 2: [C, A, B]
x2 = torch.stack([C, A, B], dim=1)  # (1, 3, 4)
out2 = simple_self_attention(x2)

print("순서 [A, B, C]에서 A의 출력:")
print(f"  {out1[0, 0].detach().numpy()}")
print("\n순서 [C, A, B]에서 A의 출력:")
print(f"  {out2[0, 1].detach().numpy()}")
print(f"\n같은가? {torch.allclose(out1[0, 0], out2[0, 1], atol=1e-6)}")
print("\n→ 위치가 바뀌어도 각 토큰의 attention 결과는 동일!")
print("→ Transformer는 순서를 모른다. 위치 정보를 별도로 넣어줘야 한다.")

---
## 2. Sinusoidal Positional Encoding (원본 논문)

Vaswani et al. (2017)이 제안한 방식. 서로 다른 주파수의 sin/cos 함수를 사용.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

- $pos$: 토큰의 위치 (0, 1, 2, ...)
- $i$: 차원 인덱스 (0, 1, 2, ..., d_model/2)

### 왜 sin/cos인가?
1. **고유성**: 각 위치마다 고유한 벡터
2. **상대 위치 표현**: $PE_{pos+k}$를 $PE_{pos}$의 선형 변환으로 표현 가능
3. **무한 확장**: 학습 데이터보다 긴 시퀀스에도 적용 가능 (이론적으로)

In [ ]:
class SinusoidalPE(nn.Module):
    """Sinusoidal Positional Encoding (Vaswani et al., 2017)"""
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # PE 행렬 계산
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # (max_len, 1)
        
        # div_term = 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )  # (d_model/2,)
        
        pe[:, 0::2] = torch.sin(position * div_term)  # 짝수 인덱스: sin
        pe[:, 1::2] = torch.cos(position * div_term)  # 홀수 인덱스: cos
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)  # 학습되지 않는 버퍼
    
    def forward(self, x):
        """x: (batch, seq_len, d_model)"""
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# PE 생성
d_model = 64
max_len = 100
pe_module = SinusoidalPE(d_model, max_len, dropout=0.0)

# PE 값 확인
pe = pe_module.pe.squeeze()  # (max_len, d_model)
print(f"PE shape: {pe.shape}")
print(f"\n위치 0의 PE: {pe[0, :8].numpy()}")
print(f"위치 1의 PE: {pe[1, :8].numpy()}")
print(f"위치 2의 PE: {pe[2, :8].numpy()}")

In [ ]:
# Sinusoidal PE 시각화 (heatmap)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: PE heatmap (전체)
ax = axes[0]
im = ax.imshow(pe[:50, :].numpy(), cmap='RdBu', aspect='auto', interpolation='nearest')
ax.set_xlabel('Dimension (i)', fontsize=12)
ax.set_ylabel('Position (pos)', fontsize=12)
ax.set_title('Sinusoidal PE Heatmap (pos 0~49)', fontsize=13)
plt.colorbar(im, ax=ax)

# 오른쪽: 특정 차원의 PE 값 (sin/cos 패턴)
ax = axes[1]
positions = range(100)
for dim in [0, 1, 4, 5, 20, 21]:
    label = f"dim {dim} ({'sin' if dim % 2 == 0 else 'cos'})"
    ax.plot(positions, pe[:100, dim].numpy(), label=label, alpha=0.7)
ax.set_xlabel('Position', fontsize=12)
ax.set_ylabel('PE Value', fontsize=12)
ax.set_title('PE Values by Dimension', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("왼쪽: 각 위치(행)마다 고유한 패턴을 가진다")
print("오른쪽: 낮은 차원은 고주파(빠르게 변화), 높은 차원은 저주파(느리게 변화)")

In [ ]:
# PE 벡터 간 유사도 (상대 위치 표현 확인)

# 위치 간 코사인 유사도 계산
pe_normalized = pe / pe.norm(dim=-1, keepdim=True)
similarity = torch.mm(pe_normalized[:50], pe_normalized[:50].T)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 위치 간 유사도 heatmap
ax = axes[0]
im = ax.imshow(similarity.numpy(), cmap='RdBu', aspect='auto')
ax.set_xlabel('Position', fontsize=12)
ax.set_ylabel('Position', fontsize=12)
ax.set_title('Position Similarity (Cosine)', fontsize=13)
plt.colorbar(im, ax=ax)

# 특정 위치(0)와의 유사도
ax = axes[1]
ax.plot(similarity[0, :50].numpy(), label='pos=0 vs others')
ax.plot(similarity[10, :50].numpy(), label='pos=10 vs others')
ax.plot(similarity[25, :50].numpy(), label='pos=25 vs others')
ax.set_xlabel('Position', fontsize=12)
ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title('Similarity with Reference Position', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("가까운 위치일수록 유사도가 높다 → 상대 위치 정보를 내포")

---
## 3. Learned Positional Embedding

위치 정보를 고정 함수가 아닌 **학습 가능한 파라미터**로 표현.

$$PE = \text{nn.Embedding}(\text{max\_len}, d_{model})$$

- BERT, GPT-2 등이 사용
- 장점: 데이터에 맞는 최적의 위치 표현을 학습
- 단점: max_len 이상의 위치에 대해 **외삽(extrapolation)** 불가

In [ ]:
class LearnedPE(nn.Module):
    """Learned Positional Embedding"""
    
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """x: (batch, seq_len, d_model)"""
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  # (1, seq_len)
        x = x + self.position_embedding(positions)
        return self.dropout(x)


# 테스트
d_model = 64
learned_pe = LearnedPE(d_model, max_len=512, dropout=0.0)

x = torch.randn(1, 10, d_model)
output = learned_pe(x)

print(f"입력 shape: {x.shape}")
print(f"출력 shape: {output.shape}")
print(f"학습 파라미터 수: {sum(p.numel() for p in learned_pe.parameters()):,}")
print(f"  = max_len({512}) x d_model({d_model}) = {512 * d_model:,}")
print(f"\n→ Sinusoidal PE: 파라미터 0개 (고정 함수)")
print(f"→ Learned PE: {512 * d_model:,}개 파라미터 (학습)")

In [ ]:
# Learned PE의 초기 상태 시각화 (학습 전이므로 랜덤)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Sinusoidal (고정)
ax = axes[0]
pe_sin = SinusoidalPE(d_model, dropout=0.0)
pe_vals = pe_sin.pe.squeeze()[:50].numpy()
im = ax.imshow(pe_vals, cmap='RdBu', aspect='auto')
ax.set_title('Sinusoidal PE (fixed)', fontsize=13)
ax.set_xlabel('Dimension')
ax.set_ylabel('Position')
plt.colorbar(im, ax=ax)

# Learned (초기 랜덤)
ax = axes[1]
learned_vals = learned_pe.position_embedding.weight[:50].detach().numpy()
im = ax.imshow(learned_vals, cmap='RdBu', aspect='auto')
ax.set_title('Learned PE (random init)', fontsize=13)
ax.set_xlabel('Dimension')
ax.set_ylabel('Position')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("Sinusoidal: 규칙적인 패턴 (학습 불필요)")
print("Learned: 랜덤 초기화 → 학습을 통해 최적화됨")

---
## 4. Rotary Position Embedding (RoPE)

현대 LLM (LLaMA, Qwen, Mistral 등)에서 가장 많이 사용되는 방식.

### 핵심 아이디어
- 위치 정보를 **회전(rotation)** 으로 인코딩
- Q, K 벡터를 위치에 따라 회전시킴
- 두 위치의 내적이 **상대 위치**에만 의존하게 됨

### 수식

2차원 벡터 $[q_0, q_1]$에 위치 $m$을 인코딩:

$$\begin{pmatrix} q_0' \\ q_1' \end{pmatrix} = \begin{pmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{pmatrix} \begin{pmatrix} q_0 \\ q_1 \end{pmatrix}$$

여기서 $\theta_i = 10000^{-2i/d}$

### 장점
- **상대 위치** 정보를 내적에 자연스럽게 반영
- **외삽 성능**이 좋음 (학습 시보다 긴 시퀀스에서도 작동)
- 추가 파라미터 없음

In [ ]:
class RotaryPE(nn.Module):
    """Rotary Position Embedding (RoPE)"""
    
    def __init__(self, d_model, max_len=5000, base=10000):
        super().__init__()
        self.d_model = d_model
        
        # theta_i = base^(-2i/d_model)
        theta = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('theta', theta)
    
    def forward(self, x, seq_len):
        """
        RoPE를 Q 또는 K에 적용
        x: (batch, n_heads, seq_len, d_k)
        """
        positions = torch.arange(seq_len, device=x.device).float()  # (seq_len,)
        
        # 각 위치, 각 차원의 각도 계산
        # (seq_len,) x (d_model/2,) → (seq_len, d_model/2)
        angles = positions.unsqueeze(1) * self.theta.unsqueeze(0)
        
        # cos, sin 계산
        cos_vals = torch.cos(angles)  # (seq_len, d_model/2)
        sin_vals = torch.sin(angles)
        
        # x를 짝수/홀수 차원으로 분리
        x_even = x[..., 0::2]  # (batch, heads, seq, d_k/2)
        x_odd = x[..., 1::2]
        
        # 회전 적용
        # x_even' = x_even * cos - x_odd * sin
        # x_odd'  = x_even * sin + x_odd * cos
        cos_vals = cos_vals.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, d_k/2)
        sin_vals = sin_vals.unsqueeze(0).unsqueeze(0)
        
        x_rotated_even = x_even * cos_vals - x_odd * sin_vals
        x_rotated_odd = x_even * sin_vals + x_odd * cos_vals
        
        # 다시 합치기
        x_rotated = torch.stack([x_rotated_even, x_rotated_odd], dim=-1)
        x_rotated = x_rotated.flatten(-2)  # (batch, heads, seq, d_k)
        
        return x_rotated


# 테스트
d_k = 8
rope = RotaryPE(d_k)

# 예시 Q, K
q = torch.randn(1, 1, 4, d_k)  # (batch=1, head=1, seq=4, d_k=8)
k = torch.randn(1, 1, 4, d_k)

q_rotated = rope(q, seq_len=4)
k_rotated = rope(k, seq_len=4)

print(f"원본 Q[0,0,0]: {q[0,0,0].detach().numpy().round(3)}")
print(f"회전 Q[0,0,0]: {q_rotated[0,0,0].detach().numpy().round(3)}")
print(f"\n→ 같은 벡터가 위치에 따라 다르게 회전됨")
print(f"→ 벡터의 노름(크기)은 보존: {q[0,0,0].norm():.4f} → {q_rotated[0,0,0].norm():.4f}")

In [ ]:
# RoPE의 핵심 특성: 내적이 상대 위치에만 의존

d_k = 32
rope = RotaryPE(d_k)

# 같은 벡터를 서로 다른 위치에 놓기
torch.manual_seed(42)
v = torch.randn(1, 1, 1, d_k)  # 하나의 벡터

# 여러 위치에서의 벡터
v_repeated = v.repeat(1, 1, 20, 1)  # 20개 위치에 같은 벡터
v_rotated = rope(v_repeated, seq_len=20)

# 위치 간 내적 계산 (RoPE 적용 후)
dot_products = torch.matmul(v_rotated, v_rotated.transpose(-2, -1)).squeeze()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 위치 간 내적 heatmap
ax = axes[0]
im = ax.imshow(dot_products.detach().numpy(), cmap='RdBu', aspect='auto')
ax.set_xlabel('Position j')
ax.set_ylabel('Position i')
ax.set_title('RoPE: Dot Product between Positions', fontsize=13)
plt.colorbar(im, ax=ax)

# 특정 위치에서 다른 위치와의 내적
ax = axes[1]
for ref_pos in [0, 5, 10]:
    ax.plot(dot_products[ref_pos].detach().numpy(), label=f'pos={ref_pos}')
ax.set_xlabel('Position')
ax.set_ylabel('Dot Product')
ax.set_title('Dot Product from Reference Position', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("모든 곡선이 같은 모양 → 내적이 상대 위치(거리)에만 의존")
print("가까운 위치일수록 내적이 크다 → 위치 정보가 attention에 반영됨")

---
## 5. ALiBi (Attention with Linear Biases)

PE를 임베딩에 더하는 대신, **attention score에 위치 기반 bias**를 추가.

$$\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + m \cdot \text{bias}\right)$$

bias는 두 위치 간 거리에 비례하는 음수:
```
bias[i][j] = -|i - j|
```

$m$은 head마다 다른 기울기 (geometric sequence).

### 장점
- **외삽(extrapolation) 성능이 매우 좋다**: 학습 길이의 몇 배까지도 잘 작동
- 구현이 간단
- 추가 파라미터 없음

In [ ]:
class ALiBi(nn.Module):
    """ALiBi: Attention with Linear Biases"""
    
    def __init__(self, n_heads, max_len=2048):
        super().__init__()
        self.n_heads = n_heads
        
        # Head마다 다른 기울기 m (geometric sequence)
        # m = 2^(-8/n_heads), 2^(-16/n_heads), ...
        closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))
        base = 2 ** (-(2 ** -(math.log2(closest_power_of_2) - 3)))
        slopes = torch.tensor([
            base ** (i + 1) for i in range(n_heads)
        ])
        self.register_buffer('slopes', slopes)  # (n_heads,)
    
    def get_bias(self, seq_len):
        """
        위치 기반 bias 생성
        Returns: (1, n_heads, seq_len, seq_len)
        """
        # 거리 행렬: |i - j|
        positions = torch.arange(seq_len)
        distance = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs().float()  # (seq_len, seq_len)
        
        # 각 head별 기울기 적용
        # slopes: (n_heads,) → (n_heads, 1, 1)
        bias = -distance.unsqueeze(0) * self.slopes.unsqueeze(1).unsqueeze(2)  # (n_heads, seq_len, seq_len)
        
        return bias.unsqueeze(0)  # (1, n_heads, seq_len, seq_len)


# 테스트
n_heads = 8
seq_len = 20
alibi = ALiBi(n_heads)

bias = alibi.get_bias(seq_len)
print(f"Bias shape: {bias.shape}")
print(f"Head별 기울기 (slopes): {alibi.slopes.numpy().round(4)}")

In [ ]:
# ALiBi bias 시각화

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

b = bias.squeeze().numpy()  # (n_heads, seq_len, seq_len)

for i in range(n_heads):
    im = axes[i].imshow(b[i], cmap='RdBu_r', aspect='auto')
    axes[i].set_title(f'Head {i+1} (m={alibi.slopes[i]:.4f})', fontsize=10)
    axes[i].set_xlabel('Key pos')
    axes[i].set_ylabel('Query pos')
    plt.colorbar(im, ax=axes[i])

plt.suptitle('ALiBi Bias: Head별 다른 기울기', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Head 1: 급경사 → 매우 가까운 토큰에만 집중 (local attention)")
print("Head 8: 완경사 → 먼 토큰도 볼 수 있음 (global attention)")
print("\n→ Head마다 다른 범위의 context를 보도록 유도")

---
## 6. 각 방식 비교

### 비교 테이블

| 방식 | 도입 | 위치 반영 위치 | 학습 파라미터 | 외삽 성능 | 사용 모델 |
|------|------|---------------|-------------|----------|----------|
| **Sinusoidal** | 2017 | 임베딩에 더함 | 없음 | 보통 | 원본 Transformer |
| **Learned** | 2018 | 임베딩에 더함 | max_len * d | 나쁨 | BERT, GPT-2 |
| **RoPE** | 2021 | Q, K를 회전 | 없음 | 좋음 | LLaMA, Qwen, Mistral |
| **ALiBi** | 2021 | Attention score에 bias | 없음 | 매우 좋음 | BLOOM, MPT |

In [ ]:
# 외삽(Extrapolation) 성능 시뮬레이션
# 학습 시 max_len=50, 추론 시 100까지 확장했을 때

train_len = 50
test_len = 100
d_model = 64

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Sinusoidal PE - 외삽 가능 (함수이므로)
ax = axes[0]
sinpe = SinusoidalPE(d_model, max_len=200, dropout=0.0)
pe_vals = sinpe.pe.squeeze()
sim = torch.mm(
    pe_vals[:test_len] / pe_vals[:test_len].norm(dim=-1, keepdim=True),
    (pe_vals[:test_len] / pe_vals[:test_len].norm(dim=-1, keepdim=True)).T
)
im = ax.imshow(sim.numpy(), cmap='RdBu', aspect='auto')
ax.axvline(x=train_len, color='red', linestyle='--', label=f'Train limit ({train_len})')
ax.axhline(y=train_len, color='red', linestyle='--')
ax.set_title('Sinusoidal PE\n(smooth extrapolation)', fontsize=11)
ax.legend(fontsize=8)

# 2. Learned PE - 외삽 불가 (학습 범위 밖은 정의되지 않음)
ax = axes[1]
learned = LearnedPE(d_model, max_len=train_len, dropout=0.0)
lpe = learned.position_embedding.weight.detach()
sim_l = torch.mm(
    lpe / lpe.norm(dim=-1, keepdim=True),
    (lpe / lpe.norm(dim=-1, keepdim=True)).T
)
# 학습 범위만 표시
full_sim = torch.zeros(test_len, test_len)
full_sim[:train_len, :train_len] = sim_l
im = ax.imshow(full_sim.numpy(), cmap='RdBu', aspect='auto')
ax.axvline(x=train_len, color='red', linestyle='--', label=f'Train limit ({train_len})')
ax.axhline(y=train_len, color='red', linestyle='--')
ax.set_title('Learned PE\n(NO extrapolation)', fontsize=11)
ax.legend(fontsize=8)

# 3. ALiBi - 외삽 우수 (거리 기반이므로)
ax = axes[2]
alibi_test = ALiBi(n_heads=1)
bias_full = alibi_test.get_bias(test_len).squeeze()
# bias를 softmax에 넣었을 때 확률 분포 시뮬레이션
im = ax.imshow(bias_full[0].numpy(), cmap='RdBu_r', aspect='auto')
ax.axvline(x=train_len, color='red', linestyle='--', label=f'Train limit ({train_len})')
ax.axhline(y=train_len, color='red', linestyle='--')
ax.set_title('ALiBi\n(excellent extrapolation)', fontsize=11)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Sinusoidal: 학습 범위 밖에서도 패턴이 연속적 (함수 기반)")
print("Learned: 학습 범위 밖은 정의 안 됨 (IndexError)")
print("ALiBi: 거리 기반이므로 자연스럽게 확장 (가장 좋은 외삽 성능)")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Sinusoidal PE의 상대 위치 특성 증명

Sinusoidal PE의 핵심 특성: $PE_{pos+k}$가 $PE_{pos}$의 **선형 변환**으로 표현 가능.

즉, 어떤 행렬 $M_k$가 존재하여:
$$PE_{pos+k} = M_k \cdot PE_{pos}$$

이를 코드로 검증하세요.

힌트:
- 2차원 (sin, cos 한 쌍)에서 $M_k = \begin{pmatrix} \cos(k\theta) & \sin(k\theta) \\ -\sin(k\theta) & \cos(k\theta) \end{pmatrix}$
- $PE_0$와 $M_k$를 곱해서 $PE_k$가 되는지 확인

In [ ]:
# TODO: 2차원에서 선형 변환 특성 검증
# 1. d_model=2 인 Sinusoidal PE를 만들기
# 2. PE[0]과 회전 행렬 M_k를 곱해서 PE[k]와 같은지 확인
# 3. 여러 k (1, 5, 10, 20)에 대해 검증


### 연습 2: RoPE와 Sinusoidal PE의 위치 유사도 비교

같은 d_model에서 RoPE와 Sinusoidal PE가 만드는 위치 유사도 패턴을 비교 시각화하세요.

- Sinusoidal: $\text{cosine\_similarity}(PE_i, PE_j)$ 계산
- RoPE: 동일한 벡터를 각 위치에서 rotate한 후 내적 계산
- 두 heatmap을 나란히 그리고 차이점을 분석

In [ ]:
# TODO:
# 1. d_model=32, seq_len=50으로 설정
# 2. Sinusoidal PE의 위치 간 코사인 유사도 행렬 계산
# 3. RoPE로 동일 벡터를 회전 후 내적 행렬 계산
# 4. 두 heatmap을 나란히 시각화


---
## 핵심 정리

| 개념 | 핵심 포인트 |
|------|------------|
| 위치 정보 필요성 | Transformer는 순서를 모름 → 별도로 주입 필요 |
| Sinusoidal PE | sin/cos 함수, 파라미터 없음, 원본 논문 |
| Learned PE | nn.Embedding, 학습 최적화, 외삽 불가 |
| RoPE | Q,K 회전, 상대 위치, 현대 LLM 표준 |
| ALiBi | Attention bias, 외삽 최강, 구현 간단 |
| 외삽(Extrapolation) | 학습 길이 이상에서의 성능: ALiBi > RoPE > Sin > Learned |

**다음 노트북**: [04-bert-vs-gpt.ipynb](04-bert-vs-gpt.ipynb) - BERT vs GPT 아키텍처 비교